# Food Delivery Business Performance — Visualization Portfolio

**Day 14 Assignment | Matplotlib & Seaborn**

This notebook builds a visualization portfolio for the Food Delivery Business Performance Dataset.  
The charts are selected to investigate:

- Orders and revenue trends over time
- Differences between cities and cuisines
- Marketing spend vs. revenue
- Delivery time vs. customer ratings
- Weather and order-channel effects on performance
- Distributions, outliers, categorical variation, and numerical correlations

Each visualization is followed by a short, data-driven interpretation.


## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Load the dataset
possible_paths = [
    "Day14_Food_Delivery_Visualization_Dataset.csv",
    "/mnt/data/Day14_Food_Delivery_Visualization_Dataset.csv"
]

csv_path = next((p for p in possible_paths if __import__("os").path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except Exception:
        raise FileNotFoundError("Please upload the Food Delivery Business Performance Dataset.")

df = pd.read_csv(csv_path)
df["Date"] = pd.to_datetime(df["Date"])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")


## 2. Initial Inspection

In [ ]:
display(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing Values"))
print("\nDuplicate rows:", df.duplicated().sum())


## 3. Summary Statistics

A quick descriptive summary helps establish the scale and spread of the main business variables before visualization.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
display(df[numeric_cols].describe().T)


# Visualization Portfolio

## 4. Line Plot — Orders and Revenue Over Time

A line chart is appropriate for identifying changes and peaks across the time dimension.


In [ ]:
daily = (
    df.groupby("Date", as_index=False)
      .agg(Orders=("Orders", "sum"), Revenue=("Revenue", "sum"))
      .sort_values("Date")
)

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(daily["Date"], daily["Orders"], marker="o", linewidth=1.8, label="Orders")
ax1.set_title("Daily Orders Over Time", fontsize=16, fontweight="bold")
ax1.set_xlabel("Date")
ax1.set_ylabel("Orders")
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(daily["Date"], daily["Revenue"], marker="o", linewidth=1.8, label="Revenue")
ax2.set_ylabel("Revenue (₹)")

fig.tight_layout()
plt.show()


**Interpretation:** Orders and revenue fluctuate across the observed period, with higher-order days generally producing higher revenue. The two series should be read using their separate y-axes because orders and revenue are measured on very different scales.


## 5. Bar Chart — Average Revenue by City

A bar chart makes category-to-category performance differences easy to compare.


In [ ]:
city_summary = (
    df.groupby("City")
      .agg(Orders=("Orders","mean"), Revenue=("Revenue","mean"))
      .sort_values("Revenue", ascending=False)
)

plt.figure(figsize=(11, 6))
sns.barplot(data=city_summary.reset_index(), x="City", y="Revenue")
plt.title("Average Revenue by City", fontsize=16, fontweight="bold")
plt.xlabel("City")
plt.ylabel("Average Revenue (₹)")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


**Interpretation:** Revenue varies noticeably by city. The ranking helps identify stronger and weaker geographic markets and can guide deeper investigation into local demand, pricing, cuisine mix, or customer behavior.


## 6. Bar Chart — Average Revenue by Cuisine

In [ ]:
cuisine_summary = (
    df.groupby("Cuisine")
      .agg(Orders=("Orders","mean"), Revenue=("Revenue","mean"))
      .sort_values("Revenue", ascending=False)
)

plt.figure(figsize=(11, 6))
sns.barplot(data=cuisine_summary.reset_index(), x="Cuisine", y="Revenue")
plt.title("Average Revenue by Cuisine", fontsize=16, fontweight="bold")
plt.xlabel("Cuisine")
plt.ylabel("Average Revenue (₹)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


**Interpretation:** Different cuisines generate different average revenue levels. This can indicate differences in average order value, demand, or the mix of cities and channels where each cuisine is sold.


## 7. Scatter Plot — Marketing Spend vs Revenue

A scatter plot is suitable for exploring whether two continuous business variables move together.


In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(
    data=df,
    x="Marketing_Spend",
    y="Revenue",
    scatter_kws={"alpha": 0.55},
    line_kws={"linewidth": 2}
)
plt.title("Marketing Spend vs Revenue", fontsize=16, fontweight="bold")
plt.xlabel("Marketing Spend (₹)")
plt.ylabel("Revenue (₹)")
plt.tight_layout()
plt.show()


**Interpretation:** The chart shows a positive overall relationship: observations with greater marketing spend tend to have higher revenue. However, the spread around the trend line shows that marketing spend alone does not explain all revenue variation, and the association should not be interpreted as proof that marketing directly caused the revenue increase.


## 8. Scatter Plot — Delivery Time vs Customer Rating

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(
    data=df,
    x="Avg_Delivery_Minutes",
    y="Customer_Rating",
    scatter_kws={"alpha": 0.55},
    line_kws={"linewidth": 2}
)
plt.title("Delivery Time vs Customer Rating", fontsize=16, fontweight="bold")
plt.xlabel("Average Delivery Time (minutes)")
plt.ylabel("Customer Rating")
plt.tight_layout()
plt.show()


**Interpretation:** The downward trend is strong: longer delivery times are associated with lower customer ratings. This is one of the clearest operational relationships in the dataset and suggests that delivery speed is closely linked with customer experience.


## 9. Histogram — Revenue Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="Revenue", bins=25, kde=True)
plt.title("Distribution of Revenue", fontsize=16, fontweight="bold")
plt.xlabel("Revenue (₹)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


**Interpretation:** The histogram shows how frequently different revenue levels occur and whether the distribution is concentrated or spread out. The KDE curve provides a smoothed view of the overall shape.


## 10. Histogram — Orders Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="Orders", bins=20, kde=True)
plt.title("Distribution of Orders", fontsize=16, fontweight="bold")
plt.xlabel("Orders")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


**Interpretation:** Most observations fall within a central range of order volumes, while the tails represent unusually low or high demand periods. This helps identify the normal operating range for order volume.


## 11. Box Plot — Revenue by City

Box plots compare distributions while also showing the median, spread, and potential outliers.


In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(data=df, x="City", y="Revenue")
plt.title("Revenue Distribution by City", fontsize=16, fontweight="bold")
plt.xlabel("City")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


**Interpretation:** The box plot shows that city performance differs not only in average revenue but also in variability. Wider boxes or longer whiskers indicate greater variation in revenue across observations, while isolated points may represent unusually strong or weak periods.


## 12. Violin Plot — Customer Ratings by Weather

A violin plot combines category comparison with the shape of each distribution.


In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x="Weather", y="Customer_Rating", inner="box")
plt.title("Customer Rating Distribution by Weather", fontsize=16, fontweight="bold")
plt.xlabel("Weather")
plt.ylabel("Customer Rating")
plt.tight_layout()
plt.show()


**Interpretation:** Weather conditions produce visibly different rating distributions. Rainy conditions are associated with lower ratings overall, while clearer conditions tend to have higher ratings, consistent with the delivery-time pattern.


## 13. Count Plot — Order Channels

A count plot is useful for comparing how frequently observations occur across categorical groups.


In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x="Order_Channel", order=df["Order_Channel"].value_counts().index)
plt.title("Number of Observations by Order Channel", fontsize=16, fontweight="bold")
plt.xlabel("Order Channel")
plt.ylabel("Number of Observations")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


**Interpretation:** The count plot shows the relative representation of App, Website, and Partner Platform observations. This is important context when comparing channel performance because unequal sample sizes can affect aggregate results.


## 14. Count Plot — Weather Conditions

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x="Weather", order=df["Weather"].value_counts().index)
plt.title("Number of Observations by Weather Condition", fontsize=16, fontweight="bold")
plt.xlabel("Weather")
plt.ylabel("Number of Observations")
plt.tight_layout()
plt.show()


**Interpretation:** The count plot shows how frequently each weather condition appears in the dataset. This helps distinguish a genuine performance pattern from one driven by a very small number of observations.


## 15. Weather and Performance — Average Orders and Revenue

In [ ]:
weather_summary = (
    df.groupby("Weather")
      .agg(
          Orders=("Orders","mean"),
          Revenue=("Revenue","mean"),
          Delivery_Time=("Avg_Delivery_Minutes","mean"),
          Rating=("Customer_Rating","mean")
      )
      .round(2)
)

display(weather_summary)

plot_df = weather_summary.reset_index().melt(
    id_vars="Weather",
    value_vars=["Orders", "Revenue"],
    var_name="Metric",
    value_name="Average"
)

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="Weather", y="Average", hue="Metric")
plt.title("Average Orders and Revenue by Weather", fontsize=16, fontweight="bold")
plt.xlabel("Weather")
plt.ylabel("Average Value")
plt.tight_layout()
plt.show()


**Interpretation:** Weather affects the operating environment. Rainy periods show the weakest customer experience and the longest delivery times, while order/revenue differences across weather conditions are smaller than the delivery-time and rating differences.


## 16. Order Channel Performance — Average Orders and Revenue

In [ ]:
channel_summary = (
    df.groupby("Order_Channel")
      .agg(
          Orders=("Orders","mean"),
          Revenue=("Revenue","mean"),
          Avg_Order_Value=("Average_Order_Value","mean"),
          Rating=("Customer_Rating","mean")
      )
      .round(2)
)

display(channel_summary)

plot_df = channel_summary.reset_index().melt(
    id_vars="Order_Channel",
    value_vars=["Orders", "Revenue"],
    var_name="Metric",
    value_name="Average"
)

plt.figure(figsize=(11, 6))
sns.barplot(data=plot_df, x="Order_Channel", y="Average", hue="Metric")
plt.title("Average Orders and Revenue by Order Channel", fontsize=16, fontweight="bold")
plt.xlabel("Order Channel")
plt.ylabel("Average Value")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


**Interpretation:** The App channel records the highest average order volume and revenue among the available channels. This makes the App an important channel for both demand generation and revenue performance in this dataset.


## 17. Correlation Heatmap

The Pearson correlation matrix is used to identify linear relationships among numerical variables.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)
plt.title("Correlation Heatmap of Numerical Variables", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


**Interpretation:** The heatmap highlights several important relationships. Orders and revenue have a strong positive relationship, while delivery time and customer rating have a very strong negative relationship. Customer rating and repeat-customer percentage also move strongly together. Correlation indicates association, not causation.


## 18. Strongest Numerical Correlations

In [ ]:
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

strongest = (
    upper.stack()
         .sort_values(key=lambda s: s.abs(), ascending=False)
         .to_frame("Pearson_Correlation")
)

display(strongest.head(12).round(3))


**Interpretation:** Ranking the unique correlation pairs makes it easier to identify the strongest relationships without being distracted by the diagonal values of 1.00.


# 19. Most Important Findings

The following conclusions are directly supported by the visualizations and correlation analysis:


1. Revenue is strongly positively related to order volume (Pearson r = 0.76), meaning periods with more orders generally generate substantially more revenue.
2. Marketing spend has a positive but relatively modest correlation with revenue (r = 0.20); higher marketing investment is associated with higher revenue, but the relationship is not strong enough to infer causation.
3. Among cities, Bengaluru has the highest average revenue per observation (₹72,678), while Kochi has the lowest (₹49,462), indicating clear geographic differences in performance.
4. Healthy cuisine produces the highest average revenue per observation (₹73,363), whereas Fast Food is lowest (₹46,820).
5. App orders have the highest average order volume (153.2 orders per observation), ahead of the other order channels.
6. Rainy weather has the longest average delivery time (38.7 minutes) and the lowest average customer rating (4.23), suggesting adverse weather is associated with weaker delivery experience.
7. Delivery time and customer rating have a very strong negative correlation (r = -0.88); as delivery time increases, ratings tend to decrease sharply.
8. Customer rating and repeat-customer percentage are strongly positively correlated (r = 0.82), indicating that better-rated observations tend to coincide with higher repeat-customer percentages.

## 20. Overall Conclusion

The visualization portfolio reveals that **business volume and customer experience are the two major patterns** in the dataset. Revenue is strongly connected with order volume, while delivery time has a pronounced negative relationship with customer ratings. Geographic location and cuisine also show meaningful differences in average revenue, indicating that market mix matters.

The App is the strongest order channel by average order volume, and weather—especially rain—is associated with longer delivery times and weaker ratings. Marketing spend has a positive relationship with revenue, but the relationship is much weaker than the order-volume/revenue relationship, so marketing should not be treated as the sole explanation for sales.

Overall, the charts provide a practical view of **when performance changes, where it is strongest, which categories differ, and which operational factors are most closely associated with customer experience**.


---
### Submission Checklist

- [x] Line plot
- [x] Bar charts
- [x] Scatter plots
- [x] Histograms
- [x] Box plot
- [x] Violin plot
- [x] Count plots
- [x] Correlation heatmap
- [x] Titles, axis labels, and formatting
- [x] Interpretation after each visualization
- [x] Final summary of findings

**GitHub submission:** Upload this `.ipynb` file to a public GitHub repository and ensure the notebook renders correctly on GitHub/Colab.
